# 여행 시뮬레이션 API 테스트

프론트엔드가 전송할 여행 일정 JSON으로 `POST /api/simulation/analyze`를 호출하고, 점수·경고·혼잡도·자차/대중교통 비교 결과를 확인합니다.

먼저 프로젝트 루트에서 백엔드를 실행하세요.

```bash
docker compose up -d backend
```

In [1]:
from pathlib import Path
import json
import os
import sys
import requests
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = next(
    (path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (path / 'docker-compose.yml').exists()),
    NOTEBOOK_DIR.parents[1],
)
BACK_DIR = PROJECT_ROOT / 'back'
load_dotenv(PROJECT_ROOT / '.env')

if str(BACK_DIR) not in sys.path:
    sys.path.insert(0, str(BACK_DIR))

BASE_URL = os.getenv('SIMULATION_API_BASE_URL', 'http://localhost:8000')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('BASE_URL:', BASE_URL)

PROJECT_ROOT: /Users/ahramkim/Documents/GitHub/route-check
BASE_URL: http://localhost:8000


## 1. 환경변수 상태 확인

보안을 위해 실제 키 값은 출력하지 않고 설정 여부만 표시합니다. `KAKAO_REST_API_KEY`는 자동차 경로, `ODSAY_API_KEY`는 대중교통 경로에 필요합니다.

In [ ]:
ENV_KEYS = {
    'TOUR_API_DECODE_KEY': '관광지 운영시간/휴무일',
    'KAKAO_REST_API_KEY': '자동차 실제 경로',
    'TMAP_API_KEY': '자동차 경로 대체 API',
    'ODSAY_API_KEY': '대중교통 실제 경로',
    'OPENAI_API_KEY': 'AI 분석 문구',
    'DATABASE_URL': 'DB 및 경로 캐시',
}

for key, purpose in ENV_KEYS.items():
    value = os.getenv(key, '').strip()
    is_placeholder = value.lower().startswith(('your_', '여기에_')) or value.lower() == 'placeholder'
    state = 'SET' if value and not is_placeholder else 'MISSING'
    print(f'{key:<24} {state:<8} # {purpose}')

## 2. 프론트엔드 요청 JSON

`transport_mode_to_next`를 생략하면 최상위 `transport_mode`가 각 이동 구간에 적용됩니다. 테스트하려는 실제 `contentid`, 경도(`mapx`), 위도(`mapy`)로 바꿔 실행하세요.

In [ ]:
payload = {
    'start_date': '2026-08-10',
    'end_date': '2026-08-10',
    'transport_mode': 'public',  # car | public | taxi | walk | bicycle
    'days': [
        {
            'day_number': 1,
            'date': '2026-08-10',
            'places': [
                {
                    'sequence': 1,
                    'contentid': 126508,
                    'title': '경복궁',
                    'mapx': 126.9770,
                    'mapy': 37.5796,
                    'stay_duration_minutes': 90,
                },
                {
                    'sequence': 2,
                    'contentid': 126509,
                    'title': '북촌한옥마을',
                    'mapx': 126.9849,
                    'mapy': 37.5826,
                    'stay_duration_minutes': 60,
                },
            ],
        }
    ],
}

print(json.dumps(payload, ensure_ascii=False, indent=2))

## 3. 요청 스키마 사전 검증

서버 호출 전 현재 백엔드 Pydantic 스키마로 JSON을 검사합니다.

In [ ]:
from schemas.simulation import SimulationRequest

validated_request = SimulationRequest.model_validate(payload)
print('스키마 검증 성공')
print(validated_request.model_dump_json(indent=2))

## 4. 서버 상태 및 시뮬레이션 실행

In [ ]:
try:
    health_response = requests.get(f'{BASE_URL}/', timeout=5)
    health_response.raise_for_status()
    print('백엔드 상태:', health_response.json())
except requests.RequestException as exc:
    raise RuntimeError(
        '백엔드에 연결할 수 없습니다. 프로젝트 루트에서 docker compose up -d backend를 실행하세요.'
    ) from exc

response = requests.post(
    f'{BASE_URL}/api/simulation/analyze',
    json=payload,
    timeout=120,
)
print('HTTP status:', response.status_code)

if not response.ok:
    print(json.dumps(response.json(), ensure_ascii=False, indent=2))
    response.raise_for_status()

result = response.json()
print(json.dumps(result, ensure_ascii=False, indent=2))

## 5. 결과 화면용 핵심 데이터 확인

In [ ]:
print('점수:', result.get('total_score'))
print('상태:', result.get('status_message'))
print('설명:', result.get('status_description'))
print('요약:', json.dumps(result.get('analysis_summary', {}), ensure_ascii=False, indent=2))

print('\n[제안 목록]')
for index, suggestion in enumerate(result.get('suggestions', []), start=1):
    print(f"{index}. [{suggestion.get('type')}] {suggestion.get('title')}")
    print('  ', suggestion.get('description'))

print('\n[경고 목록]')
for warning in result.get('warnings', []):
    print(f"- DAY {warning.get('day_number')} [{warning.get('type')}] {warning.get('message')}")

## 6. 구간별 자차/대중교통 및 혼잡도 비교

`source`가 `api` 또는 `cache`면 외부 경로 데이터 기반이고, `heuristics`면 위경도 기반 추정값입니다. 혼잡도의 `basis`가 `rule_based_estimate`이면 실시간 유동인구가 아닌 규칙 기반 예상치입니다.

In [ ]:
for day in result.get('timeline', []):
    print(f"\nDAY {day['day_number']} / {day['date']}")
    schedule = day.get('schedule', [])
    for index, place in enumerate(schedule):
        print(f"- {place['start_time']}~{place['end_time']} {place['title']}")
        congestion = place.get('congestion', {})
        print(
            f"  혼잡 예상 {congestion.get('peak_start')}~{congestion.get('peak_end')} / "
            f"방문시간 중첩={congestion.get('is_overlap')} / 근거={congestion.get('basis')}"
        )
        transit = place.get('transit_to_next')
        if not transit:
            continue
        next_title = schedule[index + 1]['title'] if index + 1 < len(schedule) else '다음 장소'
        print(
            f"  → {next_title}: 선택={transit.get('mode')}, "
            f"추천={transit.get('recommended_mode')}, source={transit.get('source')}"
        )
        for mode, alternative in transit.get('alternatives', {}).items():
            print(
                f"    {mode:<7} {alternative.get('distance_km')}km / "
                f"{alternative.get('duration_minutes')}분 / source={alternative.get('source')}"
            )